In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from datetime import datetime

SOURCE_PATH     = "/Volumes/bfsi/landing/upi/"
TABLE_NAME      = "bfsi.bronze_layer.upi_transactions"
SCHEMA          = "bronze_layer"
CHECKPOINT_PATH = f"/Volumes/bfsi/landing/upi/_checkpoint/{SCHEMA}_upi_transactions"

BATCH_ID = dbutils.widgets.get("batch_id") \
    if "batch_id" in [w.name for w in dbutils.widgets.getAll()] \
    else f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}"


In [0]:
upi_schema = StructType([
    StructField("txn_ts", TimestampType(), True),
    StructField("upi_txn_id", StringType(), False),
    StructField("payer_vpa", StringType(), True),
    StructField("payee_vpa", StringType(), True),
    StructField("payer_bank", StringType(), True),
    StructField("payee_bank", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),              # SUCCESS/FAILED/PENDING
    StructField("remarks", StringType(), True),
    StructField("device_fingerprint", StringType(), True),
    StructField("ip_address", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True)
])

In [0]:
df_upi_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("header", "true")           
    .option("delimiter", "|")           
    .schema(upi_schema)
    .load(SOURCE_PATH)
    .withColumn("_source_system", F.lit("NPCI_UPI"))
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_file_name", F.col("_metadata.file_path"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .drop("_metadata")
    )

In [0]:
query = (
    df_upi_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_NAME)
)

query.awaitTermination()

print(f"Auto Loader stream completed for {TABLE_NAME}")

In [0]:
df_verify = spark.table(TABLE_NAME)

print(f"Total UPI transactions ingested: {df_verify.count()}")
display(df_verify.limit(5))

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# ---------------------------------------------------------
# 1. CLEANUP (Optional: Only if you want to start from zero)
# ---------------------------------------------------------
spark.sql(f"TRUNCATE TABLE {TABLE_NAME}")

# ---------------------------------------------------------
# 2. READ DATA & CAPTURE METADATA
# ---------------------------------------------------------
# In Unity Catalog batch reads, we select "_metadata" explicitly
df_upi = (
    spark.read
    .format("csv")
    .option("header", "true")        # Skips the row containing column names
    .option("delimiter", "|")
    .schema(upi_schema)              # Uses your predefined schema
    .load(SOURCE_PATH)
    .select("*", "_metadata.file_path") # Extract the file path from hidden metadata
)

# ---------------------------------------------------------
# 3. TRANSFORM (Add Metadata Columns)
# ---------------------------------------------------------
df_final = (
    df_upi
    .withColumnRenamed("file_path", "_file_name") # Rename the extracted metadata column
    .withColumn("_source_system", F.lit("NPCI_UPI"))
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

# ---------------------------------------------------------
# 4. WRITE TO DELTA TABLE
# ---------------------------------------------------------
(
    df_final.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLE_NAME)
)

print(f"Successfully pushed UC-compliant batch data to {TABLE_NAME}")